# Muestreo de distribuciones estadísticas en Python

Si está trabajando en el modelado de simulación en Python, probablemente necesitará utilizar el espacio de nombres `numpy.random`. Proporciona una variedad de distribuciones estadísticas que puede utilizar para un muestreo eficiente. 

Este cuaderno le guiará a través de ejemplos de 

1. Creación de instancias de un generador de números pseudoaleatorios (PRNG) de alta calidad utilizando PCG64 proporcionado por `numpy`
2. Generar muestras a partir de las distribuciones **uniforme**, **exponencial** y **normal**.
3. Generar múltiples flujos de números aleatorios que no se superpongan
4. Uso de programación orientada a objetos para encapsular PRNG, distribuciones y parámetros para modelos de simulación.

## 1. Importaciones

Importaremos `numpy` para nuestro muestreo y `matplotlib` para trazar nuestras distribuciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2. Funciones auxiliares

La función simple siguiente se puede utilizar para producir automáticamente un gráfico que ilustra una distribución de muestras.

In [ ]:
def distribution_plot(samples, bins=100, figsize=(5,3)):
    '''
    Helper function to visualise the distributions
    
    Params:
    -----
    samples: np.ndarray
        A numpy array of quantitative data to plot as a histogram.
        
    bins: int, optional (default=100)
        The number of bins to include in the histogram
        
    figsize: (int, int) (default=(5,3))
        Size of the plot in pixels
        
    Returns:
    -------
        fig, ax: a tuple containing matplotlib figure and axis objects.
    '''
    hist = np.histogram(samples, bins=np.arange(bins), 
                        density=True)

    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot()
    _ = ax.plot(hist[0])
    _ = ax.set_ylabel('p(x)')
    _ = ax.set_xlabel('x')
    
    return fig, ax

## 3. Creando un objeto generador de números aleatorios

Para generar números pseudoaleatorios para muestreo de cada distribución, podemos usar la función `default_rng()` del módulo `numpy.random`.

Esta función construye una instancia de una clase `Generator`, que puede producir números aleatorios. 

De forma predeterminada, `numpy` utiliza un generador de números pseudoaleatorios (PRNG) llamado uso del [Generador congruente permutado de 64 bits] (https://www.pcg-random.org/) (PCG64; período = $2^{128}$; número máximo de secuencias = $2^{127}$).

Para obtener más información sobre `Generator`, puede consultar la [documentación en línea de `numpy`.](https://numpy.org/doc/stable/reference/random/generator.html)

In [ ]:
rng = np.random.default_rng()

In [ ]:
type(rng)

## 4. Pasos para crear una muestra

En general, el enfoque para el muestreo es:

1. Cree un objeto **generador** de números aleatorios

2. Usando el objeto llame al método para la **distribución estadística**
    * Cada método tiene sus propios parámetros personalizados
    * Cada método incluirá un parámetro `size` que utilizará para establecer el número de muestras a generar.

3. **Almacenar** el resultado en una variable con el nombre apropiado

### 4.1 Distribución uniforme

In [ ]:
# Step 1: create a random number generator object - set seed to 42
rng = np.random.default_rng(42)

# Step 2 and 3: call the appropriate method of the generator and store result
samples = rng.uniform(low=10, high=40, size=1_000_000)

# Illustrate with plot.
_ = distribution_plot(samples, bins=50)

### 4.2 Distribución exponencial

In [ ]:
rng = np.random.default_rng(42)
samples = rng.exponential(scale=12, size=1_000_000)
_ = distribution_plot(samples, bins=50)

## 4.3 Distribución normal

In [ ]:
rng = np.random.default_rng(42)
samples = rng.normal(loc=25.0, scale=5.0, size=1_000_000)
_ = distribution_plot(samples, bins=50)

## 4.4 Generando una única muestra

Si solo necesitamos generar una sola muestra, omitimos el parámetro `size`. Esto devuelve un valor escalar.

In [ ]:
rng = np.random.default_rng(42)
sample = rng.normal(loc=25.0, scale=5.0)
print(sample)
print(type(sample))

**Tenga en cuenta** que también puede configurar `size` en 1. Solo tenga en cuenta que se devuelve una matriz. p.ej.

In [ ]:
rng = np.random.default_rng(42)
sample = rng.normal(loc=25.0, scale=5.0, size=1)
# a numpy array is returned
print(sample)
print(type(sample))

# to access the scalar value use the 0 index of the array.
print(sample[0])

## 5. Generación de múltiples flujos PRN que no se superponen.

Para la simulación, lo ideal es utilizar múltiples flujos de números aleatorios que no se superpongan (es decir, que sean independientes). Esto es sencillo de implementar en Python usando `SeedSequence` y una semilla entera proporcionada por el usuario y la cantidad de transmisiones independientes para generar.

> Como usuario, no necesitamos preocuparnos por la calidad de la semilla entera proporcionada. Esto es útil para implementar múltiples replicaciones y números aleatorios comunes.

Así es como creamos las semillas a partir de una semilla proporcionada por un único usuario.  La variable devuelta `seeds` es una `List` de Python.

In [ ]:
n_streams = 2
user_seed = 1

seed_sequence = np.random.SeedSequence(user_seed)
seeds = seed_sequence.spawn(n_streams)

Usamos `seeds` al crear nuestros PRNG.  Por ejemplo, uno para los tiempos entre llegadas y otro para los tiempos de servicio.

In [ ]:
# e.g. to model arrival times
arrival_rng = np.random.default_rng(seeds[0])

# e.g. to model service times
service_rng = np.random.default_rng(seeds[1])

## 6. Encapsular distribuciones, parámetros y semillas aleatorias.

Al construir un modelo de simulación, suele ser útil *empaquetar* un generador de números aleatorios, parámetros para una distribución específica y una semilla en una **clase de Python**.  Esto permite una **creación sencilla** de objetos generadores, un muestreo sencillo y mejora la gestión de flujos para cada actividad en un modelo de simulación.

Como ejemplo a continuación, la clase `Exponential` representa la distribución exponencial. Acepta un parámetro de valor medio y puede configurar la semilla aleatoria.

Luego, crearemos una instancia de dos objetos `Exponential` para dos procesos diferentes en nuestra simulación: duración de la estancia aguda y duración de la estancia en rehabilitación.

In [ ]:
class Exponential:
    '''
    Convenience class for the exponential distribution.
    Packages up distribution parameters, seed and random generator.
    '''
    def __init__(self, mean, random_seed=None):
        '''
        Constructor

        Params:
        ------
        mean: float
            The mean of the exponential distribution

        random_seed: int | SeedSequence, optional (default=None)
            A random seed to reproduce samples.  If set to none then a unique
            sample is created.
        '''
        self.rand = np.random.default_rng(seed=random_seed)
        self.mean = mean

    def sample(self, size=None):
        '''
        Generate a sample from the exponential distribution

        Params:
        -------
        size: int, optional (default=None)
            the number of samples to return.  If size=None then a single
            sample is returned.
        '''
        return self.rand.exponential(self.mean, size=size)

In [ ]:
acute_los = Exponential(3.0, random_seed=42)
rehab_los = Exponential(30.0, random_seed=101)

In [ ]:
acute_los.sample()

In [ ]:
rehab_los.sample()

## 7. Próximos pasos

Ahora podemos pasar a la creación de modelos simples de simulación de eventos discretos utilizando modelos `simpy` que utilizan el muestreo `numpy`.